# Heart Disease UCI — EDA & model walkthrough
Thin notebook over the `heartml` package so the notebook and the CI pipeline run *the same code*.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
import pandas as pd
from heartml import config, data, eda

## 1. Acquire & clean

In [ ]:
raw = data.download()
df = data.clean(raw)
print(df.shape)
df.head()

In [ ]:
data.missing_report(df)

In [ ]:
df.describe().T.round(2)

## 2. EDA figures

In [ ]:
summary = eda.run(df)
from IPython.display import Image, display
for f in summary['figures']:
    display(Image(f))

In [ ]:
pd.Series(summary['top_correlations_with_target'])

## 3. Train both models (logged to MLflow)
Takes ~1 minute. Open the MLflow UI afterwards: `mlflow ui --backend-store-uri ./mlruns --port 5000`.

In [ ]:
from heartml import train
meta = train.main()
pd.DataFrame(meta['comparison']).T

## 4. Inference with the saved joblib model

In [ ]:
from heartml.predict import load_model, predict_records
sample = json.load(open('scripts/sample_request.json'))
predict_records(load_model(str(config.MODEL_FILE)), [sample])